In [1]:
import pandas as pd
import numpy as np
import os

PROCESSED_DIR = "data/processed"
RAW_DIR = "data/raw"

team_features_2026 = pd.read_csv(f"{PROCESSED_DIR}/team_features_2026.csv")
performance_history = pd.read_csv(f"{RAW_DIR}/historical/worldcup_team_performance_history.csv")

print("team_features_2026:", team_features_2026.shape)
print(list(team_features_2026.columns))

print("\nperformance_history:", performance_history.shape)
print(list(performance_history.columns))

team_features_2026: (48, 24)
['team', 'squad_size', 'average_age', 'average_height', 'total_caps', 'average_caps', 'total_goals', 'average_goals', 'number_of_goalkeepers', 'number_of_defenders', 'number_of_midfielders', 'number_of_forwards', 'team_code', 'association', 'rank', 'previous_rank', 'points', 'previous_points', 'rated_matches', 'manager_name', 'manager_nationality', 'appointment_year', 'preferred_formation', 'style_description']

performance_history: (192, 24)
['version', 'team', 'continent', 'is_host', 'goals_scored_last_4y', 'goals_received_last_4y', 'wins_last_4y', 'losses_last_4y', 'draws_last_4y', 'world_cup_titles_before', 'squad_total_market_value_eur', 'fifa_rank_pre_tournament', 'fifa_points_pre_tournament', 'squad_avg_age', 'world_cup_participations_before', 'groups_passed_before', 'round16_before', 'quarterfinals_before', 'semifinals_before', 'finals_before', 'winner', 'finalist', 'semi_finalist', 'quarter_finalist']


In [2]:
model_features = [
    "draws_last_4y_difference", "fifa_points_pre_tournament_difference",
    "fifa_rank_pre_tournament_difference", "finals_before_difference",
    "goals_received_last_4y_difference", "goals_scored_last_4y_difference",
    "groups_passed_before_difference", "is_host_difference",
    "losses_last_4y_difference", "quarterfinals_before_difference",
    "round16_before_difference", "semifinals_before_difference",
    "squad_avg_age_difference", "squad_total_market_value_eur_difference",
    "wins_last_4y_difference", "world_cup_participations_before_difference",
    "world_cup_titles_before_difference", "fifa_points_pre_tournament_ratio",
    "goals_scored_last_4y_ratio", "goals_received_last_4y_ratio",
    "wins_last_4y_ratio", "squad_avg_age_ratio", "host_difference",
]

def get_base_name(feature_name):
    if feature_name.endswith("_difference"):
        return feature_name[:-len("_difference")]
    if feature_name.endswith("_ratio"):
        return feature_name[:-len("_ratio")]
    return feature_name

required_bases = sorted(set(get_base_name(f) for f in model_features))
print(f"{len(required_bases)} unique base columns required:")
for b in required_bases:
    print(" ", b)

18 unique base columns required:
  draws_last_4y
  fifa_points_pre_tournament
  fifa_rank_pre_tournament
  finals_before
  goals_received_last_4y
  goals_scored_last_4y
  groups_passed_before
  host
  is_host
  losses_last_4y
  quarterfinals_before
  round16_before
  semifinals_before
  squad_avg_age
  squad_total_market_value_eur
  wins_last_4y
  world_cup_participations_before
  world_cup_titles_before


In [3]:
existing_2026_cols = set(team_features_2026.columns)
existing_hist_cols = set(performance_history.columns)

direct_renames = {
    "rank": "fifa_rank_pre_tournament",
    "points": "fifa_points_pre_tournament",
    "average_age": "squad_avg_age",
}
renamed_targets = set(direct_renames.values())

print("Availability check:\n")
for b in required_bases:
    if b in existing_2026_cols:
        source = "team_features_2026 (direct match)"
    elif b in renamed_targets:
        source = "team_features_2026 (via rename - see Cell 4)"
    elif b in existing_hist_cols:
        source = "performance_history (needs 2022->2026 projection)"
    else:
        source = "NOT AVAILABLE - needs documented substitute"
    print(f"  {b:35s} -> {source}")

Availability check:

  draws_last_4y                       -> performance_history (needs 2022->2026 projection)
  fifa_points_pre_tournament          -> team_features_2026 (via rename - see Cell 4)
  fifa_rank_pre_tournament            -> team_features_2026 (via rename - see Cell 4)
  finals_before                       -> performance_history (needs 2022->2026 projection)
  goals_received_last_4y              -> performance_history (needs 2022->2026 projection)
  goals_scored_last_4y                -> performance_history (needs 2022->2026 projection)
  groups_passed_before                -> performance_history (needs 2022->2026 projection)
  host                                -> NOT AVAILABLE - needs documented substitute
  is_host                             -> performance_history (needs 2022->2026 projection)
  losses_last_4y                      -> performance_history (needs 2022->2026 projection)
  quarterfinals_before                -> performance_history (needs 2022->2026 projec

In [28]:
base = team_features_2026[["team"]].copy()

for old_col, new_col in direct_renames.items():
    if old_col in team_features_2026.columns:
        base[new_col] = team_features_2026[old_col]
        print(f"Mapped '{old_col}' -> '{new_col}'")
    else:
        print(f"WARNING: '{old_col}' not found - '{new_col}' will be missing")

# Host status: FIFA officially confirmed three co-hosts for 2026.
host_nations_2026 = {"USA", "Mexico", "Canada"}

base["is_host"] = base["team"].isin(host_nations_2026).astype(int)
base["host"] = base["is_host"]  

print(f"\nHost nations flagged: {base.loc[base['is_host']==1, 'team'].tolist()}")

Mapped 'rank' -> 'fifa_rank_pre_tournament'
Mapped 'points' -> 'fifa_points_pre_tournament'
Mapped 'average_age' -> 'squad_avg_age'

Host nations flagged: ['Canada', 'Mexico', 'USA']


In [6]:
perf_2022 = performance_history[
    performance_history["version"] == 2022
].copy()

print("2022 performance_history rows:", perf_2022.shape)

2022 performance_history rows: (32, 24)


In [8]:
perf_2022 = performance_history[
    performance_history["version"] == 2022
].copy()

print("2022 performance_history rows:", perf_2022.shape)

base = base.merge(
    perf_2022,
    on="team",
    how="left",
    suffixes=("", "_2022")
)

has_2022 = base["world_cup_titles_before"].notna()

base["has_2022_history"] = has_2022.astype(int)

print(f"\nTeams WITH 2022 historical data: {has_2022.sum()}")
print(f"Teams WITHOUT 2022 data (debut nations): {(~has_2022).sum()}")

print("\nTeams without 2022 history:")
print(base.loc[~has_2022, "team"].tolist())

2022 performance_history rows: (32, 24)

Teams WITH 2022 historical data: 23
Teams WITHOUT 2022 data (debut nations): 25

Teams without 2022 history:
['Algeria', 'Austria', 'Bosnia And Herzegovina', 'Cabo Verde', 'Colombia', 'Congo DR', 'Curaçao', 'Czechia', "Côte D'Ivoire", 'Egypt', 'Haiti', 'IR Iran', 'Iraq', 'Jordan', 'Korea Republic', 'New Zealand', 'Norway', 'Panama', 'Paraguay', 'Scotland', 'South Africa', 'Sweden', 'Türkiye', 'USA', 'Uzbekistan']


In [9]:
print(sorted(perf_2022["team"].unique()))

['Argentina', 'Australia', 'Belgium', 'Brazil', 'Cameroon', 'Canada', 'Costa Rica', 'Croatia', 'Denmark', 'Ecuador', 'England', 'France', 'Germany', 'Ghana', 'Iran', 'Japan', 'Mexico', 'Morocco', 'Netherlands', 'Poland', 'Portugal', 'Qatar', 'Saudi Arabia', 'Senegal', 'Serbia', 'South Korea', 'Spain', 'Switzerland', 'Tunisia', 'United States', 'Uruguay', 'Wales']


In [10]:
name_fix = {
    "IR Iran": "Iran",
    "Korea Republic": "South Korea",
    "USA": "United States"
}

base["team"] = base["team"].replace(name_fix)

# rerun merge
perf_2022 = performance_history[
    performance_history["version"] == 2022
].copy()

base = base.drop(
    columns=[c for c in base.columns if c.endswith("_2022")],
    errors="ignore"
)

base = base.merge(
    perf_2022,
    on="team",
    how="left",
    suffixes=("", "_2022")
)

has_2022 = base["world_cup_titles_before"].notna()

print("Teams WITH 2022 historical data:", has_2022.sum())
print("Teams WITHOUT 2022 historical data:", (~has_2022).sum())

print("\nRemaining teams without 2022 history:")
print(sorted(base.loc[~has_2022, "team"].tolist()))

Teams WITH 2022 historical data: 23
Teams WITHOUT 2022 historical data: 25

Remaining teams without 2022 history:
['Algeria', 'Austria', 'Bosnia And Herzegovina', 'Cabo Verde', 'Colombia', 'Congo DR', 'Curaçao', 'Czechia', "Côte D'Ivoire", 'Egypt', 'Haiti', 'Iran', 'Iraq', 'Jordan', 'New Zealand', 'Norway', 'Panama', 'Paraguay', 'Scotland', 'South Africa', 'South Korea', 'Sweden', 'Türkiye', 'United States', 'Uzbekistan']


In [11]:
[c for c in base.columns if "world_cup" in c]

['world_cup_titles_before',
 'world_cup_participations_before',
 'world_cup_titles_before_2022',
 'world_cup_participations_before_2022']

In [12]:
print(base["world_cup_titles_before_2022"].notna().sum())

26


In [13]:
has_2022 = base["world_cup_titles_before_2022"].notna()

base["has_2022_history"] = has_2022.astype(int)

print(f"Teams WITH 2022 historical data: {has_2022.sum()}")
print(f"Teams WITHOUT 2022 historical data: {(~has_2022).sum()}")

print("\nTeams without 2022 history:")
print(sorted(base.loc[~has_2022, "team"].tolist()))

Teams WITH 2022 historical data: 26
Teams WITHOUT 2022 historical data: 22

Teams without 2022 history:
['Algeria', 'Austria', 'Bosnia And Herzegovina', 'Cabo Verde', 'Colombia', 'Congo DR', 'Curaçao', 'Czechia', "Côte D'Ivoire", 'Egypt', 'Haiti', 'Iraq', 'Jordan', 'New Zealand', 'Norway', 'Panama', 'Paraguay', 'Scotland', 'South Africa', 'Sweden', 'Türkiye', 'Uzbekistan']


In [14]:
[
    c for c in base.columns
    if "finals_before" in c
    or "semifinals_before" in c
    or "quarterfinals_before" in c
]

['quarterfinals_before',
 'semifinals_before',
 'finals_before',
 'quarterfinals_before_2022',
 'semifinals_before_2022',
 'finals_before_2022']

In [30]:
for col in pedigree_cols:
    print(col, col in base.columns)

world_cup_participations_before False
world_cup_titles_before False
finals_before False
semifinals_before False
quarterfinals_before False


In [31]:
for col in pedigree_cols:
    print(f"{col}_2022", f"{col}_2022" in base.columns)

world_cup_participations_before_2022 False
world_cup_titles_before_2022 False
finals_before_2022 False
semifinals_before_2022 False
quarterfinals_before_2022 False


In [33]:
for c in base.columns:
    if any(x in c.lower() for x in ["world", "title", "final", "semi", "quarter", "particip"]):
        print(c)

In [34]:
type(base)

pandas.core.frame.DataFrame

In [35]:
for c in base.columns:
    if any(x in c.lower() for x in [
        "world", "title", "final", "semi", "quarter", "particip"
    ]):
        print(c)

In [36]:
print(base.shape)
print(base.columns.tolist())

(48, 6)
['team', 'fifa_rank_pre_tournament', 'fifa_points_pre_tournament', 'squad_avg_age', 'is_host', 'host']


In [37]:
print("BASE SHAPE:", base.shape)

print("\n2022 columns found:")
for c in base.columns:
    if "_2022" in c:
        print(c)

BASE SHAPE: (48, 6)

2022 columns found:


In [29]:
# Starting from the actual 2022 pedigree values
pedigree_cols = [
    "world_cup_participations_before",
    "world_cup_titles_before",
    "finals_before",
    "semifinals_before",
    "quarterfinals_before",
]

for col in pedigree_cols:
    base[col] = base[f"{col}_2022"]

# Adding the results of the 2022 World Cup
base["world_cup_participations_before"] = (
    base["world_cup_participations_before"].fillna(0)
    + base["has_2022_history"]
)

base["world_cup_titles_before"] = (
    base["world_cup_titles_before"].fillna(0)
    + base["winner"].fillna(0)
)

base["finals_before"] = (
    base["finals_before"].fillna(0)
    + base["finalist"].fillna(0)
)

base["semifinals_before"] = (
    base["semifinals_before"].fillna(0)
    + base["semi_finalist"].fillna(0)
)

base["quarterfinals_before"] = (
    base["quarterfinals_before"].fillna(0)
    + base["quarter_finalist"].fillna(0)
)

print("Projected 'before 2026' pedigree values:")
print(
    base[
        [
            "team",
            "world_cup_participations_before",
            "world_cup_titles_before",
            "finals_before",
            "semifinals_before",
            "quarterfinals_before",
        ]
    ].head()
)

KeyError: 'world_cup_participations_before_2022'

In [16]:
zero_fill_cols = [
    "groups_passed_before", "round16_before",
    "goals_scored_last_4y", "goals_received_last_4y",
    "wins_last_4y", "losses_last_4y", "draws_last_4y",
]

for col in zero_fill_cols:
    n_filled = base[col].isna().sum()
    base[col] = base[col].fillna(0)
    if n_filled > 0:
        print(f"Filled {n_filled} missing value(s) in '{col}' with 0 "
              f"(debut nations - genuinely zero recorded history)")

Filled 25 missing value(s) in 'groups_passed_before' with 0 (debut nations - genuinely zero recorded history)
Filled 25 missing value(s) in 'round16_before' with 0 (debut nations - genuinely zero recorded history)
Filled 25 missing value(s) in 'goals_scored_last_4y' with 0 (debut nations - genuinely zero recorded history)
Filled 25 missing value(s) in 'goals_received_last_4y' with 0 (debut nations - genuinely zero recorded history)
Filled 25 missing value(s) in 'wins_last_4y' with 0 (debut nations - genuinely zero recorded history)
Filled 25 missing value(s) in 'losses_last_4y' with 0 (debut nations - genuinely zero recorded history)
Filled 25 missing value(s) in 'draws_last_4y' with 0 (debut nations - genuinely zero recorded history)


In [17]:
for col in [
    "groups_passed_before",
    "round16_before",
    "goals_scored_last_4y",
    "wins_last_4y"
]:
    print(f"\n{col}")
    print(base[base[col].isna()][["team"]])


groups_passed_before
Empty DataFrame
Columns: [team]
Index: []

round16_before
Empty DataFrame
Columns: [team]
Index: []

goals_scored_last_4y
Empty DataFrame
Columns: [team]
Index: []

wins_last_4y
Empty DataFrame
Columns: [team]
Index: []


In [18]:
# squad_total_market_value_eur is unavailable from any current source:
# - team_summary has no market-value column
# - performance_history's value reflects 2022 squads, not 2026
#
# Rather than substitute a different statistic AS IF it were market
# value, we set this to a constant for every team. This makes
# squad_total_market_value_eur_difference == 0 for every 2026 matchup
# - an honest "we have no information here" rather than a fabricated one.
#
# v2: populate from Transfermarkt squad valuations.

base["squad_total_market_value_eur"] = 0
print("squad_total_market_value_eur set to constant 0 for all 48 teams.")
print("-> squad_total_market_value_eur_difference = 0 for every 2026 matchup.")

squad_total_market_value_eur set to constant 0 for all 48 teams.
-> squad_total_market_value_eur_difference = 0 for every 2026 matchup.


In [19]:
required_cols = [
    "draws_last_4y",
    "fifa_points_pre_tournament",
    "fifa_rank_pre_tournament",
    "finals_before",
    "goals_received_last_4y",
    "goals_scored_last_4y",
    "groups_passed_before",
    "host",
    "is_host",
    "losses_last_4y",
    "quarterfinals_before",
    "round16_before",
    "semifinals_before",
    "squad_avg_age",
    "squad_total_market_value_eur",
    "wins_last_4y",
    "world_cup_participations_before",
    "world_cup_titles_before"
]

missing = [c for c in required_cols if c not in base.columns]

print("Missing required columns:")
print(missing if missing else "None")

Missing required columns:
None


In [20]:
cols_to_drop = [c for c in base.columns if c.endswith("_2022")]
cols_to_drop += ["winner", "finalist", "semi_finalist", "quarter_finalist", "year"]
cols_to_drop = [c for c in cols_to_drop if c in base.columns]

base = base.drop(columns=cols_to_drop)

print(f"Dropped {len(cols_to_drop)} helper columns: {cols_to_drop}")
print(f"\nFinal table shape: {base.shape}")
print("\nColumns:")
print(list(base.columns))

Dropped 27 helper columns: ['version_2022', 'continent_2022', 'is_host_2022', 'goals_scored_last_4y_2022', 'goals_received_last_4y_2022', 'wins_last_4y_2022', 'losses_last_4y_2022', 'draws_last_4y_2022', 'world_cup_titles_before_2022', 'squad_total_market_value_eur_2022', 'fifa_rank_pre_tournament_2022', 'fifa_points_pre_tournament_2022', 'squad_avg_age_2022', 'world_cup_participations_before_2022', 'groups_passed_before_2022', 'round16_before_2022', 'quarterfinals_before_2022', 'semifinals_before_2022', 'finals_before_2022', 'winner_2022', 'finalist_2022', 'semi_finalist_2022', 'quarter_finalist_2022', 'winner', 'finalist', 'semi_finalist', 'quarter_finalist']

Final table shape: (48, 22)

Columns:
['team', 'fifa_rank_pre_tournament', 'fifa_points_pre_tournament', 'squad_avg_age', 'is_host', 'host', 'version', 'continent', 'goals_scored_last_4y', 'goals_received_last_4y', 'wins_last_4y', 'losses_last_4y', 'draws_last_4y', 'world_cup_titles_before', 'squad_total_market_value_eur', 'wor

In [21]:
print(base.isnull().sum()[base.isnull().sum() > 0])

version      25
continent    25
dtype: int64


In [22]:
base = base.drop(
    columns=["version", "continent", "has_2022_history"],
    errors="ignore"
)

In [23]:
print(base.shape)
print(base.isnull().sum().sum())

(48, 19)
0


In [25]:
print("VALIDATION REPORT")
print("=" * 50)

print(f"\nTotal teams: {len(base)} (expected 48)")
print(f"Unique teams: {base['team'].nunique()}")

print("\nRequired base columns present?")
for b in required_bases:
    present = b in base.columns
    print(f"  {b:35s} {'OK' if present else 'MISSING'}")

print("\nMissing values per column:")
missing = base.isnull().sum()
missing = missing[missing > 0]
print(missing if not missing.empty else "  (none)")


print(f"\nHost nations (n={base['is_host'].sum()}):")
print(base.loc[base["is_host"] == 1, "team"].tolist())

VALIDATION REPORT

Total teams: 48 (expected 48)
Unique teams: 48

Required base columns present?
  draws_last_4y                       OK
  fifa_points_pre_tournament          OK
  fifa_rank_pre_tournament            OK
  finals_before                       OK
  goals_received_last_4y              OK
  goals_scored_last_4y                OK
  groups_passed_before                OK
  host                                OK
  is_host                             OK
  losses_last_4y                      OK
  quarterfinals_before                OK
  round16_before                      OK
  semifinals_before                   OK
  squad_avg_age                       OK
  squad_total_market_value_eur        OK
  wins_last_4y                        OK
  world_cup_participations_before     OK
  world_cup_titles_before             OK

Missing values per column:
  (none)

Host nations (n=3):
['Canada', 'Mexico', 'United States']


In [26]:
output_path = f"{PROCESSED_DIR}/team_features_2026_model_ready.csv"
base.to_csv(output_path, index=False)

print(f"Saved {base.shape[0]} rows x {base.shape[1]} columns to:")
print(f"  {output_path}")

Saved 48 rows x 19 columns to:
  data/processed/team_features_2026_model_ready.csv


In [27]:
print("=" * 60)
print("NOTEBOOK 03.5 - ASSUMPTIONS & APPROXIMATIONS LOG")
print("=" * 60)
print("""
REAL, DIRECT 2026 DATA (no approximation):
  fifa_rank_pre_tournament    <- fifa_ranking_2026 'rank'
  fifa_points_pre_tournament  <- fifa_ranking_2026 'points'
  squad_avg_age                <- team_summary 'average_age'
  is_host / host                <- official FIFA 2026 host list
                                    (USA, Mexico, Canada)

PROJECTED FROM REAL 2022 RESULT (one-tournament projection):
  world_cup_participations_before, world_cup_titles_before,
  finals_before, semifinals_before, quarterfinals_before
      = 'before 2022' value + 2022's actual outcome.
        Debut nations = 0 (factually correct).

CARRIED FORWARD UNCHANGED (documented v1 approximation):
  groups_passed_before, round16_before
      = 'before 2022' value (slight undercount for some
        2022 participants). Debut nations = 0.

  goals_scored_last_4y, goals_received_last_4y, wins_last_4y,
  losses_last_4y, draws_last_4y
      = 2022 snapshot (reflects 2018-2022 form, now stale).
        Debut nations = 0.
        v2: replace with 2022-2026 recent match results.

PLACEHOLDER (no source data exists):
  squad_total_market_value_eur = 0 for all teams
      -> squad_total_market_value_eur_difference = 0 everywhere
        v2: populate from Transfermarkt.
""")

print(f"Output file: {PROCESSED_DIR}/team_features_2026_model_ready.csv")
print(f"Shape: {base.shape}")

NOTEBOOK 03.5 - ASSUMPTIONS & APPROXIMATIONS LOG

REAL, DIRECT 2026 DATA (no approximation):
  fifa_rank_pre_tournament    <- fifa_ranking_2026 'rank'
  fifa_points_pre_tournament  <- fifa_ranking_2026 'points'
  squad_avg_age                <- team_summary 'average_age'
  is_host / host                <- official FIFA 2026 host list
                                    (USA, Mexico, Canada)

PROJECTED FROM REAL 2022 RESULT (one-tournament projection):
  world_cup_participations_before, world_cup_titles_before,
  finals_before, semifinals_before, quarterfinals_before
      = 'before 2022' value + 2022's actual outcome.
        Debut nations = 0 (factually correct).

CARRIED FORWARD UNCHANGED (documented v1 approximation):
  groups_passed_before, round16_before
      = 'before 2022' value (slight undercount for some
        2022 participants). Debut nations = 0.

  goals_scored_last_4y, goals_received_last_4y, wins_last_4y,
  losses_last_4y, draws_last_4y
      = 2022 snapshot (reflects 